In [1]:
!pip install nltk

In [2]:
import torch
import pandas as pd
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

from utils import clean, tokenise, load_sets, load_vocab, load_config

from encoder import Encoder
from decoder import Decoder
from seq2Seq import Seq2Seq

### Loading Test Data and Vocab

In [13]:
train_set, val_set, test_set = load_sets()

vocab = load_vocab(rev=False)
vocab_reversed = load_vocab(rev=True)
config = load_config()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
model_a = torch.load("models/model_a/model_a_full.pt", map_location=device, weights_only=False)
model_b = torch.load("models/model_b/model_b_full.pt", map_location=device, weights_only=False)

model_a.to(device)
model_b.to(device)

model_a.eval()
model_b.eval()

print("Model A and Model B loaded.")

Model A and Model B loaded.


### Token Id's to Text

In [15]:
from utils import encode_sequence

SPECIAL_TOKEN_STRINGS = {"<PAD>", "<SOS>", "<EOS>", "<UNK>"}

def ids_to_text(token_ids, vocab_reversed):
    pieces = []

    for idx in token_ids:
        token = vocab_reversed.get(str(int(idx)), None)

        if token is None:
            token = vocab_reversed.get(int(idx), "")

        if token and token not in SPECIAL_TOKEN_STRINGS:
            pieces.append(token)

    # Reconstruct text from SentencePiece pieces
    text = "".join(pieces).replace("▁", " ").strip()
    return " ".join(text.split())

### Greedy Decoding

In [16]:
def greedy_decode(model, text, vocab, vocab_reversed, config, device):
    model.eval()

    encoder_input = torch.tensor(
        encode_sequence(clean(text), vocab, config["MAX_LENGTH"]),
        dtype=torch.long
    ).unsqueeze(0).to(device)

    generated_ids = []

    with torch.no_grad():
        if getattr(model.encoder, "use_attention", False):
            encoder_outputs, hidden = model.encoder(encoder_input)
        else:
            hidden = model.encoder(encoder_input)
            encoder_outputs = None

        next_token = torch.tensor([config["SOS_IDX"]], dtype=torch.long).to(device)

        for _ in range(config["MAX_LENGTH"]):
            if getattr(model.decoder, "use_attention", False):
                pred, hidden = model.decoder(next_token, hidden, encoder_outputs)
            else:
                pred, hidden = model.decoder(next_token, hidden)

            next_token = pred.argmax(dim=-1)
            token_id = next_token.item()

            if token_id == config["EOS_IDX"]:
                break

            generated_ids.append(token_id)

    return ids_to_text(generated_ids, vocab_reversed)

### BLEU Evaluation

In [17]:
def evaluate_bleu(model, test_df, vocab, vocab_reversed, config, device, max_samples=None):
    smooth = SmoothingFunction().method1

    references = []
    hypotheses = []
    rows = []

    eval_df = test_df.copy()
    if max_samples is not None:
        eval_df = eval_df.iloc[:max_samples]

    for _, row in eval_df.iterrows():
        source_text = str(row["input"])
        reference_text = str(row["response"])

        prediction = greedy_decode(
            model=model,
            text=source_text,
            vocab=vocab,
            vocab_reversed=vocab_reversed,
            config=config,
            device=device
        )

        ref_tokens = tokenise(clean(reference_text))
        pred_tokens = tokenise(clean(prediction))

        references.append([ref_tokens])
        hypotheses.append(pred_tokens)

        sent_bleu = sentence_bleu(
            [ref_tokens],
            pred_tokens,
            smoothing_function=smooth
        )

        rows.append({
            "input": source_text,
            "reference": reference_text,
            "prediction": prediction,
            "sentence_bleu": sent_bleu
        })

    corpus_score = corpus_bleu(
        references,
        hypotheses,
        smoothing_function=smooth
    )

    results_df = pd.DataFrame(rows)
    return corpus_score, results_df

### Run Evaluation

In [18]:
# Evaluate Model A
bleu_a, results_a = evaluate_bleu(
    model=model_a,
    test_df=test_set,
    vocab=vocab,
    vocab_reversed=vocab_reversed,
    config=config,
    device=device,
    max_samples=500
)

# Evaluate Model B
bleu_b, results_b = evaluate_bleu(
    model=model_b,
    test_df=test_set,
    vocab=vocab,
    vocab_reversed=vocab_reversed,
    config=config,
    device=device,
    max_samples=500
)

print(f"Model A BLEU: {bleu_a:.4f}")
print(f"Model B BLEU: {bleu_b:.4f}")

Model A BLEU: 0.0000
Model B BLEU: 0.0000


### Comparison Table

In [19]:
comparison_df = pd.DataFrame({
    "input": results_a["input"],
    "reference": results_a["reference"],
    "model_a_prediction": results_a["prediction"],
    "model_a_sentence_bleu": results_a["sentence_bleu"],
    "model_b_prediction": results_b["prediction"],
    "model_b_sentence_bleu": results_b["sentence_bleu"],
})

comparison_df.head(10)

,input,reference,model_a_prediction,model_a_sentence_bleu,model_b_prediction,model_b_sentence_bleu
0,help! make ubuntu detect my monitors!,what is your monitor specs give the specs and...,you,0.000000e+00,you,0.000000e+00
1,re-directs command output,ok thx,i,0.000000e+00,you,0.000000e+00
2,stupid question: hoe does one change directory...,use quotes or use *,,0.000000e+00,-r,0.000000e+00
3,what is the name of the default window manager...,metacity metacity is the wm,->,0.000000e+00,is,3.257032e-03
4,crash1 - use keys,is there a site that tells me how? i googled b...,i,1.348391e-10,i,1.348391e-10
5,is there a redmond theam for gnome tux only o...,you can probably find windows-like skins,,0.000000e+00,is,0.000000e+00
6,yap froward im ms cleo,"look into my future and tell me, how does my i...",,0.000000e+00,,0.000000e+00
7,uh whats the cmd to install a .deb file,scroll up a few lines ;) please don't repeat,-i-i,0.000000e+00,dpkg-i,0.000000e+00
8,hi all irc newbie here looking for answer to ...,so use feisty!,alis,0.000000e+00,is,0.000000e+00
9,"hello, can i play warcraft in ubuntu?",i think using wine or using vm,you,0.000000e+00,youinstall,0.000000e+00
